# Example 4.4 / 4: Time Periods: Inventory Balance

Companion notebook for **4.4 Ordered Sets and Time Periods** in the AMPL lab Documentation, section "Time Periods: Inventory Balance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Time Periods: Inventory Balance

The multiperiod production model makes and sells two steel products over four weeks, and
carries inventory from week to week. Its balance constraint says that what is made plus what
was in inventory at the end of the week before equals what is sold plus what is put into
inventory this week. The week before `t` is `prev(t)`, but the first week has none: it starts
from the initial inventory `inv0[p]` instead. So the first week gets a constraint of its own,
`Balance0`, written with `first(WEEKS)`, and `Balance` is indexed over the other weeks only,
`{p in PROD, t in WEEKS: ord(t) > 1}`.

A dictionary with `(product, week)` keys gives the two-dimensional parameters their values.
After the solve, `ampl.get_data("Make", "Inv")` returns both variables in one table, and
`to_pandas()` turns it into a DataFrame:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
set WEEKS ordered;                # weeks, in order
param rate {PROD} > 0;            # tons per hour produced
param inv0 {PROD} >= 0;           # inventory before the first week
param avail {WEEKS} >= 0;         # hours available in each week
param market {PROD, WEEKS} >= 0;  # most tons that can be sold
param prodcost {PROD} >= 0;       # cost per ton produced
param invcost {PROD} >= 0;        # cost per ton held in inventory
param revenue {PROD, WEEKS} >= 0; # revenue per ton sold

var Make {PROD, WEEKS} >= 0;      # tons produced
var Inv {PROD, WEEKS} >= 0;       # tons in inventory at the end of the week
var Sell {p in PROD, t in WEEKS} >= 0, <= market[p,t];   # tons sold

maximize Total_Profit:
    sum {p in PROD, t in WEEKS}
        (revenue[p,t] * Sell[p,t] - prodcost[p] * Make[p,t] - invcost[p] * Inv[p,t]);

subject to Time {t in WEEKS}:
    sum {p in PROD} (1 / rate[p]) * Make[p,t] <= avail[t];

subject to Balance0 {p in PROD}:
    Make[p,first(WEEKS)] + inv0[p] = Sell[p,first(WEEKS)] + Inv[p,first(WEEKS)];

subject to Balance {p in PROD, t in WEEKS: ord(t) > 1}:
    Make[p,t] + Inv[p,prev(t)] = Sell[p,t] + Inv[p,t];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
weeks = ["27sep", "04oct", "11oct", "18oct"]
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["WEEKS"] = weeks
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["inv0"] = {"bands": 10, "coils": 0}
ampl.param["prodcost"] = {"bands": 10, "coils": 11}
ampl.param["invcost"] = {"bands": 2.5, "coils": 3}
ampl.param["avail"] = dict(zip(weeks, [40, 40, 32, 40]))
ampl.param["revenue"] = {("bands", t): r for t, r in zip(weeks, [25, 26, 27, 27])} | {
    ("coils", t): r for t, r in zip(weeks, [30, 35, 37, 39])}
ampl.param["market"] = {("bands", t): m for t, m in zip(weeks, [6000, 6000, 4000, 6500])} | {
    ("coils", t): m for t, m in zip(weeks, [4000, 2500, 3500, 4200])}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.get_data("Make", "Inv").to_pandas())

HiGHS 1.14.0: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations
solved
               Make   Inv
index0 index1            
bands  27sep   5990     0
       04oct   6000     0
       11oct   1400     0
       18oct   2000     0
coils  27sep   1407  1100
       04oct   1400     0
       11oct   3500     0
       18oct   4200     0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. In the data file, a table gives the
two-dimensional parameters, with the weeks as its columns:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set WEEKS := 27sep 04oct 11oct 18oct;

param:  rate  inv0  prodcost  invcost :=
  bands  200   10      10       2.5
  coils  140    0      11       3 ;

param avail := 27sep 40  04oct 40  11oct 32  18oct 40;

param revenue:  27sep  04oct  11oct  18oct :=
  bands          25     26     27     27
  coils          30     35     37     39 ;

param market:  27sep  04oct  11oct  18oct :=
  bands        6000   6000   4000   6500
  coils        4000   2500   3500   4200 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result;
display Make, Inv;

HiGHS 1.14.0: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations
solve_result = solved

:             Make   Inv     :=
bands 27sep   5990      0
bands 04oct   6000      0
bands 11oct   1400      0
bands 18oct   2000      0
coils 27sep   1407   1100
coils 04oct   1400      0
coils 11oct   3500      0
coils 18oct   4200      0
;

